# Customer Segmentation for a Large Retail Company

This is dataset used in this notebook is a modification of a Kaggle dataset and modified by Dataquest.

I decided to use this dataset to practice the following scenario:
Working as a data practitioner for the online department of a large retail company, I need to work on a customer segmentation.

The goal of this task is to determine which segments would increase sales the most and target them with ads in social media. This is an a examples of **proxies** since the request is urgent. 

The dataset has only three columns, which you immediately find familiar from your role at Floormart:

- ```customer_id:```    Customer identification number.
- ```trans_date:```     Transaction date.
- ```tran_amount:```    Transaction amount.

In [18]:
# Import libraries
import pandas as pd
import datetime as dt
from sklearn.preprocessing import MinMaxScaler


In [5]:
# Load the data
df = pd.read_csv("rfm_xmas19.txt", parse_dates=['trans_date'])

# Overview of the data
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125000 entries, 0 to 124999
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   customer_id  125000 non-null  object        
 1   trans_date   125000 non-null  datetime64[ns]
 2   tran_amount  125000 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 2.9+ MB


,customer_id,trans_date,tran_amount
0,FM5295,2017-11-11,35
1,FM4768,2019-12-15,39
2,FM2122,2017-11-26,52
3,FM1217,2016-08-16,99
4,FM1850,2018-08-20,78


## Churned Customers

A churned customer is one who has'nt purchased anything since September 16.

In [10]:
df_customer = df.groupby('customer_id')

churn = pd.DataFrame(df_customer['trans_date'].max())

cutoff_day = dt.datetime(2019, 9, 16)

churn['churned'] = churn['trans_date'].apply(lambda date: 1 if date < cutoff_day else 0)

churn

,trans_date,churned
customer_id,,
FM1112,2019-10-14,0
FM1113,2019-11-09,0
FM1114,2019-11-12,0
FM1115,2019-12-05,0
FM1116,2019-05-25,1
...,...,...
FM8996,2019-09-09,1
FM8997,2019-03-28,1
FM8998,2019-09-22,0


## Adding Number of Transactions and Amount Spent:

In [17]:
churn['transactions'] = df_customer.size()
churn['amount_spent'] = df_customer['tran_amount'].sum()
churn.drop('trans_date', axis=1, inplace=True)
churn

,churned,transactions,amount_spent
customer_id,,,
FM1112,0,15,1012
FM1113,0,20,1490
FM1114,0,19,1432
FM1115,0,22,1659
FM1116,1,13,857
...,...,...,...
FM8996,1,13,582
FM8997,1,14,543
FM8998,0,13,624


## Ranking Customers

Due to time constraints, I used a **weighted sum model** to classify customers.

<code> f(x) = (0.5) * Number of purchases + (0.5) * Amount spent </code>

However, I need to **normalize** or **min-max feature scaling** the number of purchases and amount spent since they have different scales.


In [24]:
# Maximun and minimum values
churn[['transactions', 'amount_spent']].describe().loc[['min', 'max']]

,transactions,amount_spent
min,4.0,149.0
max,39.0,2933.0


In [21]:
scaler = MinMaxScaler()
churn[['transactions_scaled', 'amount_spent_scaled']] = scaler.fit_transform(churn[['transactions', 'amount_spent']])
churn

,churned,transactions,amount_spent,transactions_scaled,amount_spent_scaled
customer_id,,,,,
FM1112,0,15,1012,0.314286,0.309986
FM1113,0,20,1490,0.457143,0.481681
FM1114,0,19,1432,0.428571,0.460848
FM1115,0,22,1659,0.514286,0.542385
FM1116,1,13,857,0.257143,0.254310
...,...,...,...,...,...
FM8996,1,13,582,0.257143,0.155532
FM8997,1,14,543,0.285714,0.141523
FM8998,0,13,624,0.257143,0.170618


In [25]:
# Max and min values of the scaled values
churn[['transactions_scaled', 'amount_spent_scaled']].describe().loc[['min', 'max']]

,transactions_scaled,amount_spent_scaled
min,0.0,0.0
max,1.0,1.0


In [29]:
churn['score'] = round(((0.5 * churn['amount_spent_scaled']) + (0.5 * churn['transactions_scaled'])) * 100, 2)
churn.sort_values('score', ascending=False, inplace=True)
churn

,churned,transactions,amount_spent,transactions_scaled,amount_spent_scaled,score
customer_id,,,,,,
FM4424,0,39,2933,1.000000,1.000000,100.00
FM4320,0,38,2647,0.971429,0.897270,93.43
FM3799,1,36,2513,0.914286,0.849138,88.17
FM5109,0,35,2506,0.885714,0.846624,86.62
FM3805,1,35,2453,0.885714,0.827586,85.67
...,...,...,...,...,...,...
FM7716,1,4,221,0.000000,0.025862,1.29
FM7224,1,4,191,0.000000,0.015086,0.75
FM8504,0,4,190,0.000000,0.014727,0.74


## Determining a Threshold
I need to determine which customers are the best customers.

I could use advanced techniques like **k-means clustering**, but these techniques take a lot of time.

For this reason, I consider the following factors:
- The budget is $1,000.
- The coupons need to be good enough to prompt people to actually use them.
- They can't be too high because:
    - That reduces the number of customers who get them.
    - It would be like giving away money.
    - Due to price dumping, it could be illegal.
- 30% discount is already very enticing.

I decided to make the following calculation to determine the value of the coupon and the number of customers:

In [37]:
coupon = df['tran_amount'].mean() * 0.3
num_customers = 1000 / coupon

print(f"Coupon Value:        {coupon:>5.0f}")
print(f"Number of Customers: {num_customers:>5.0f}")

Coupon Value:           19
Number of Customers:    51


Based on this, the coupon might be for $20 and send to the top 50 churned customers.

In [40]:
top_customers = churn[churn['churned'] == 1][:50]
top_customers

,churned,transactions,amount_spent,transactions_scaled,amount_spent_scaled,score
customer_id,,,,,,
FM3799,1,36,2513,0.914286,0.849138,88.17
FM3805,1,35,2453,0.885714,0.827586,85.67
FM4074,1,34,2462,0.857143,0.830819,84.40
FM1215,1,35,2362,0.885714,0.794899,84.03
FM2620,1,35,2360,0.885714,0.794181,83.99
FM1580,1,33,2329,0.828571,0.783046,80.58
FM2951,1,32,2382,0.800000,0.802083,80.10
FM5868,1,31,2260,0.771429,0.758261,76.48
FM1695,1,32,2174,0.800000,0.727371,76.37
